In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option("display.max_columns", 200)

# Load from the same CSV (DVC doesn’t change this code)
df = pd.read_csv("../data/raw/insurance_data.csv", sep="|", engine="python")


In [5]:
df['TransactionMonth'] = pd.to_datetime(df['TransactionMonth'], errors='coerce')

df['LossRatio'] = np.where(
    df['TotalPremium'] > 0,
    df['TotalClaims'] / df['TotalPremium'],
    np.nan
)

df['Margin'] = df['TotalPremium'] - df['TotalClaims']
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

df.head()

,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,MaritalStatus,Gender,Country,Province,PostalCode,MainCrestaZone,SubCrestaZone,ItemType,mmcode,VehicleType,RegistrationYear,make,Model,Cylinders,cubiccapacity,kilowatts,bodytype,NumberOfDoors,VehicleIntroDate,CustomValueEstimate,AlarmImmobiliser,TrackingDevice,CapitalOutstanding,NewVehicle,WrittenOff,Rebuilt,Converted,CrossBorder,NumberOfVehiclesInFleet,SumInsured,TermFrequency,CalculatedPremiumPerTerm,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims,LossRatio,Margin,HasClaim
0,145249,12827,2015-03-01,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0,2597.0,130.0,S/D,4.0,6/2002,119300.0,Yes,No,119300,More than 6 months,NaN,NaN,NaN,NaN,NaN,0.01,Monthly,25.0000,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,0.0,21.929825,0
1,145249,12827,2015-05-01,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0,2597.0,130.0,S/D,4.0,6/2002,119300.0,Yes,No,119300,More than 6 months,NaN,NaN,NaN,NaN,NaN,0.01,Monthly,25.0000,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,0.0,21.929825,0
2,145249,12827,2015-07-01,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0,2597.0,130.0,S/D,4.0,6/2002,119300.0,Yes,No,119300,More than 6 months,NaN,NaN,NaN,NaN,NaN,0.01,Monthly,25.0000,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0,NaN,0.000000,0
3,145255,12827,2015-05-01,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0,2597.0,130.0,S/D,4.0,6/2002,119300.0,Yes,No,119300,More than 6 months,NaN,NaN,NaN,NaN,NaN,119300.00,Monthly,584.6468,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0,0.0,512.848070,0
4,145255,12827,2015-07-01,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0,2597.0,130.0,S/D,4.0,6/2002,119300.0,Yes,No,119300,More than 6 months,NaN,NaN,NaN,NaN,NaN,119300.00,Monthly,584.6468,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0,NaN,0.000000,0


In [6]:
# Drop rows with missing LossRatio or Province
df_prov = df.dropna(subset=['LossRatio', 'Province'])

# Build list of groups (one array per province)
groups = []
labels = []

for prov, g in df_prov.groupby("Province"):
    vals = g['LossRatio'].dropna()
    if len(vals) > 30:  # keep provinces with enough data
        groups.append(vals)
        labels.append(prov)

len(groups), labels


(9,
 ['Eastern Cape',
  'Free State',
  'Gauteng',
  'KwaZulu-Natal',
  'Limpopo',
  'Mpumalanga',
  'North West',
  'Northern Cape',
  'Western Cape'])

In [7]:
f_stat, p_value = stats.f_oneway(*groups)
f_stat, p_value


(np.float64(4.969184128835032), np.float64(3.5625050155955674e-06))

In [8]:
# Only keep rows with LossRatio and PostalCode
df_zip = df.dropna(subset=['LossRatio', 'PostalCode']).copy()

# Count policies per zipcode
zip_counts = df_zip['PostalCode'].value_counts()

# Keep reasonably frequent zipcodes (e.g., >= 500 policies)
valid_zips = zip_counts[zip_counts >= 500].index

len(valid_zips), valid_zips[:10]


(223,
 Index([2000, 122, 299, 7784, 7405, 8000, 458, 2196, 470, 4360], dtype='int64', name='PostalCode'))

In [9]:
groups = []
labels = []

for z in valid_zips:
    vals = df_zip.loc[df_zip['PostalCode'] == z, 'LossRatio'].dropna()
    if len(vals) > 30:   # safety check
        groups.append(vals)
        labels.append(z)

len(groups), labels[:10]


(223, [2000, 122, 299, 7784, 7405, 8000, 458, 2196, 470, 4360])

In [10]:
f_stat_zip, p_value_zip = stats.f_oneway(*groups)
f_stat_zip, p_value_zip


(np.float64(1.565120543411354), np.float64(1.4582656612183308e-07))

In [11]:
df_zip_margin = df.dropna(subset=['Margin', 'PostalCode']).copy()

groups_margin = []
labels_margin = []

for z in valid_zips:
    vals = df_zip_margin.loc[df_zip_margin['PostalCode'] == z, 'Margin'].dropna()
    if len(vals) > 30:
        groups_margin.append(vals)
        labels_margin.append(z)

len(groups_margin), labels_margin[:10]


(223, [2000, 122, 299, 7784, 7405, 8000, 458, 2196, 470, 4360])

In [12]:
f_stat_margin, p_value_margin = stats.f_oneway(*groups_margin)
f_stat_margin, p_value_margin


(np.float64(1.26016234124689), np.float64(0.005166270826021837))

In [13]:
df_gender = df.dropna(subset=['LossRatio', 'Gender']).copy()

female_lr = df_gender.loc[df_gender['Gender'] == 'Female', 'LossRatio']
male_lr   = df_gender.loc[df_gender['Gender'] == 'Male', 'LossRatio']

len(female_lr), len(male_lr)


(3404, 19083)

In [14]:
t_stat_lr, p_value_lr = stats.ttest_ind(
    female_lr,
    male_lr,
    equal_var=False  # Welch's t-test (safer)
)
t_stat_lr, p_value_lr, female_lr.mean(), male_lr.mean()


(np.float64(0.579788980612858),
 np.float64(0.5620917714414586),
 np.float64(0.4919788674989967),
 np.float64(0.35325803158527924))

In [15]:
df_gender_claim = df.dropna(subset=['HasClaim', 'Gender']).copy()

ct = pd.crosstab(df_gender_claim['Gender'], df_gender_claim['HasClaim'])
ct


HasClaim,0,1
Gender,,
Female,6741,14
Male,42723,94
Not specified,938324,2666


In [16]:
chi2, p_value_chi, dof, expected = stats.chi2_contingency(ct)
chi2, p_value_chi


(np.float64(7.255926312995721), np.float64(0.026570248768437138))